# Views only: taxonomy v0.11 against v0.10

**Goal:** compare the two taxonomies on the same 4 samples, with haiku and opus.

| | v0.10 | v0.11 |
|---|---|---|
| Claim types | RECOMMENDATION, PREDICTION, CAUSAL, VIEW, OBSERVATION, SUPPOSITION | Same, **without OBSERVATION** |
| Grounds | `rationale`: OBSERVATION statements | Evidence quotes on the claims themselves |
| Subclaims | `supporting` | `premises` [P] |
| Atomicity | One meaningful assertion per statement | Statements are distinct only if one could be true while the other is false; a restated point becomes one statement |
| Argument with only a core claim | Error | Warning |

Pipeline (repo root): `.venv\Scripts\python.exe scripts\run_samples.py --limit 4`. It defaults to v0.11 and writes to `outputs/samples_v11`; v0.10 is in `outputs/samples_v10`.

**Questions**
1. Validity, cost and size: does dropping observations cut output and failures?
2. Duplication: is the same point still spread over several claims?
3. Hierarchy: arguments, links, bare arguments, motivated recommendations.
4. Coverage: are any v0.10 claims (not observations) lost, especially must-keep ones?
5. Do haiku and opus agree better on fewer, cleaner claims?
6. Grounding: do claims now cite the data as evidence?
7. Suppositions and scenarios.

The probes in this notebook are heuristics.

In [ ]:
import json
import re
import sys
from collections import Counter
from itertools import combinations
from pathlib import Path

import pandas as pd
from IPython.display import display

from incolor.argumentation.evaluate import match_statements
from incolor.argumentation.validate import argument_edges, scenario_edges
from incolor.argumentation.writer import load_argument_map

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RUN_DIRS = {
    "v0.9": ROOT / "outputs" / "samples_v09",
    "v0.10": ROOT / "outputs" / "samples_v10",
    "v0.11": ROOT / "outputs" / "samples_v11",
}
COMPARED = ["v0.10", "v0.11"]
TYPES = ["RECOMMENDATION", "PREDICTION", "CAUSAL", "VIEW", "OBSERVATION", "SUPPOSITION"]
MODELS = ["haiku", "opus"]
JSON_COLUMNS = ["evidence_ids", "evidence_texts", "roles", "argument_ids"]
sys.path.insert(0, str(ROOT / "scripts"))
import build_claims_csv  # noqa: E402


def load_claims(runs_dir):
    """Statement table for one batch, rebuilt when the set of valid maps has changed."""
    data_dir = runs_dir / "claims_analysis"
    path = data_dir / "claims_all.csv"
    valid = {p.relative_to(ROOT).as_posix() for p in runs_dir.glob("*/*/argument_map.yaml")}
    built = set(pd.read_csv(path, usecols=["map_path"], encoding="utf-8-sig")["map_path"]) if path.exists() else set()
    if built != valid:
        build_claims_csv.main([
            "--runs-root", str(runs_dir), "--texts-dir", str(runs_dir / "inputs"), "--no-gold",
            "--pivot-model", "opus", "--out", str(data_dir),
        ])
    df = pd.read_csv(path, encoding="utf-8-sig")
    for col in JSON_COLUMNS:
        df[col] = df[col].map(lambda v: json.loads(v) if isinstance(v, str) and v else [])
    df["sub"] = df["subkinds"].fillna("").str.replace("|", "+")
    return df


DOCS = sorted({p.parent.parent.name for p in RUN_DIRS["v0.11"].glob("*/*/summary.json")})
LABEL = {doc: doc.split("_-_")[0].replace("__", " ").replace("_", " ")[:24] for doc in DOCS}
claims = pd.concat({v: load_claims(RUN_DIRS[v]) for v in COMPARED}, names=["taxonomy"]).reset_index(level=0)
claims = claims[claims["doc"].isin(DOCS)].reset_index(drop=True)
claims["label"] = claims["doc"].map(LABEL)
maps = {
    v: {(p.parent.parent.name, p.parent.name): load_argument_map(p) for p in sorted(RUN_DIRS[v].glob("*/*/argument_map.yaml"))
        if p.parent.parent.name in DOCS}
    for v in RUN_DIRS
}
SHOW = ["taxonomy", "label", "model", "statement_id", "claim_type", "sub", "modality", "statement"]
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)
pd.crosstab([claims["label"], claims["model"]], claims["taxonomy"]).reindex(columns=COMPARED, fill_value=0)

samples_v10/BC_FX_Views_-_Structural_JPY_weakness_unchanged/haiku   86 statements   17 exclusions
samples_v10/BC_FX_Views_-_Structural_JPY_weakness_unchanged/opus  140 statements    9 exclusions
samples_v10/Beam_yields_up__Scotty/haiku       126 statements   21 exclusions
samples_v10/Beam_yields_up__Scotty/opus        132 statements   15 exclusions
samples_v10/BofA_Liquid_Insight_-_France_risks_for_EUR/haiku   46 statements    6 exclusions
samples_v10/BofA_Liquid_Insight_-_France_risks_for_EUR/opus   44 statements    5 exclusions
samples_v10/BofA_Sept_FOMC_-_hike_or_bond_spike/opus  115 statements    5 exclusions

7 maps -> 689 statements, 78 exclusions in C:\Users\angel.adeoye\dev\financial_research\outputs\samples_v10\claims_analysis
samples_v11/BC_FX_Views_-_Structural_JPY_weakness_unchanged/opus   79 statements    3 exclusions
samples_v11/Beam_yields_up__Scotty/haiku        72 statements   19 exclusions
samples_v11/Beam_yields_up__Scotty/opus         75 statements    8 exclusions
s

taxonomy                     v0.10  v0.11
label                 model              
BC FX Views           haiku     86      0
                      opus     140     79
Beam yields up Scotty haiku    126     72
                      opus     132     75
BofA Liquid Insight   haiku     46     26
                      opus      44     26
BofA Sept FOMC        haiku      0     81
                      opus     115     91

## 1. Runs: validity, cost and size

In [3]:
def load_runs(runs_dir):
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(runs_dir.glob("*/*/summary.json"))]
    runs = pd.DataFrame(rows)
    runs = runs[runs["doc"].isin(DOCS)].copy()
    runs["label"] = runs["doc"].map(LABEL)
    return runs


runs = pd.concat({v: load_runs(d) for v, d in RUN_DIRS.items()}, names=["taxonomy"]).reset_index(level=0)
RUN_COLS = ["ok", "attempts", "cost_usd", "seconds", "arguments", "statements", "evidence", "edges", "isolated", "exclusions"]
display(
    runs[runs["taxonomy"].isin(COMPARED)]
    .pivot_table(index=["label", "model"], columns="taxonomy", values=RUN_COLS, aggfunc="first")
    .reindex(columns=COMPARED, level=1)
)
totals = runs.groupby(["taxonomy", "model"]).agg(
    runs=("doc", "size"), valid=("ok", "sum"), cost_usd=("cost_usd", "sum"), mean_attempts=("attempts", "mean"),
    statements=("statements", "sum"), evidence=("evidence", "sum"), arguments=("arguments", "sum"),
    exclusions=("exclusions", "sum"), median_seconds=("seconds", "median"),
).round(2)
display(totals.reindex(["v0.9", "v0.10", "v0.11"], level=0))
failed = runs[~runs["ok"].fillna(False).astype(bool) & runs["taxonomy"].isin(COMPARED)]
display(failed[["taxonomy", "label", "model", "attempts", "errors", "issue_codes"]])
# Issue codes on the final attempt of each run.
codes = runs[runs["taxonomy"].isin(COMPARED)].set_index(["taxonomy", "model"])["issue_codes"].dropna().apply(pd.Series)
codes.groupby(level=[0, 1]).sum().T.fillna(0).astype(int)

arguments       attempts       cost_usd          \
taxonomy                        v0.10 v0.11    v0.10 v0.11    v0.10   v0.11   
label                 model                                                   
BC FX Views           haiku        14    17        3     3   0.1712  0.2890   
                      opus         15    17        2     1   2.7908  1.2869   
Beam yields up Scotty haiku        32    26        3     2   0.2482  0.0802   
                      opus         26    29        2     1   2.2589  1.1875   
BofA Liquid Insight   haiku         9    10        2     1   0.0582  0.0330   
                      opus          9     6        1     1   0.7530  0.5456   
BofA Sept FOMC        haiku        12    16        3     3   0.1117  0.1512   
                      opus         21    23        2     1   2.0593  1.5845   

                            edges       evidence       exclusions        \
taxonomy                    v0.10 v0.11    v0.10 v0.11      v0.10 v0.11   
label                 model                                               
BC FX Views           haiku     4    16       77    91         17    11   
                      opus     11    10      118   105          9     3   
Beam yields up Scotty haiku    15     8      106    91         21    19   
                      opus     14    17      114   101         15     8   
BofA Liquid Insight   haiku     6     7       38    32          6    20   
                      opus      5     2       44    36          5     3   
BofA Sept FOMC        haiku     3    10       97    68          9    10   
                      opus     12    10      114   120          5     7   

                            isolated           ok        seconds         \
taxonomy                       v0.10 v0.11  v0.10  v0.11   v0.10  v0.11   
label                 model                                               
BC FX Views           haiku        7     1   True  False   984.3  704.2   
                      opus         2     5   True   True   824.9  416.7   
Beam yields up Scotty haiku       13    15   True   True   627.7  488.1   
                      opus         8     9   True   True   688.8  400.7   
BofA Liquid Insight   haiku        2     1   True   True   355.9  211.7   
                      opus         1     3   True   True   222.8  150.1   
BofA Sept FOMC        haiku        7     4  False   True   619.1  928.7   
                      opus         9    11   True   True   624.4  564.2   

                            statements        
taxonomy                         v0.10 v0.11  
label                 model                   
BC FX Views           haiku         86    78  
                      opus         140    79  
Beam yields up Scotty haiku        126    72  
                      opus         132    75  
BofA Liquid Insight   haiku         46    26  
                      opus          44    26  
BofA Sept FOMC        haiku        111    81  
                      opus         115    91

runs  valid  cost_usd  mean_attempts  statements  evidence  \
taxonomy model                                                               
v0.9     haiku     4      3      0.56           2.00         327       306   
         opus      4      4      7.83           1.75         399       375   
v0.10    haiku     4      3      0.59           2.75         369       318   
         opus      4      4      7.86           1.75         431       390   
v0.11    haiku     4      3      0.55           2.25         257       282   
         opus      4      4      4.60           1.00         271       362   

                arguments  exclusions  median_seconds  
taxonomy model                                         
v0.9     haiku         74          48          570.05  
         opus          88          40          660.55  
v0.10    haiku         67          53          623.40  
         opus          71          34          656.60  
v0.11    haiku         69          60          596.15  
         opus          75          21          408.70

,taxonomy,label,model,attempts,errors,issue_codes
6,v0.10,BofA Sept FOMC,haiku,3,2,{'EVIDENCE_NOT_VERBATIM': 2}
0,v0.11,BC FX Views,haiku,3,2,"{'EVIDENCE_NOT_VERBATIM': 2, 'EVIDENCE_UNUSED': 1}"


taxonomy                  v0.10      v0.11     
model                     haiku opus haiku opus
MUST_KEEP_EXCLUDED            0    1     2    0
EVIDENCE_NOT_VERBATIM         2    0     2    0
EVIDENCE_UNUSED               0    0     2    0
ARGUMENT_WITHOUT_RELATION     0    0     2    4
EXCLUSION_NOT_VERBATIM        0    0     1    0

### Observations

- **Validity is unchanged: haiku 3/4, opus 4/4.** The haiku failure moved from FOMC (v0.10) to BC FX (v0.11), both times with `EVIDENCE_NOT_VERBATIM` after 3 attempts. BC FX haiku is still the fragile case.
- **Opus is cheaper and faster.** It cost $4.60 instead of $7.86 (−41%), passed every run on the first attempt (mean attempts 1.75 before), and took a median 409s instead of 657s. Haiku cost about the same ($0.55 vs $0.59), with mean attempts down from 2.75 to 2.25.
- **Output is smaller.** Opus produced 271 statements instead of 431 (−37%), but evidence fell only 7% (390 → 362), so the data stayed as evidence. On the two docs where haiku was valid in both versions (Beam, BofA LI), its statements went from 172 to 98.
- **Exclusions:** opus fell from 34 to 21. Haiku rose on BofA LI (6 → 20), probably because it now excludes data points it would previously have kept as observations.
- **New final-attempt issues:** `ARGUMENT_WITHOUT_RELATION` (bare-argument warning) appears 2× for haiku and 4× for opus, and `MUST_KEEP_EXCLUDED` 2× for haiku. Neither blocks validity.

## 2. Duplication: one point spread over several claims

Near-duplicate pairs within a map: two claims (not suppositions) whose content words overlap with a Jaccard score of at least 0.4. The rate is the number of pairs per 10 claims. Next, the BofA case: how many claims each version spends on "France risk is priced".

In [4]:
STOP = set(
    "the a an and or of to in on for with by at from is are was were be been will would could may might should "
    "that this these those it its as than more less into over under after before also has have had not no".split()
)


def content_words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP and len(w) > 2}


rows, pairs = [], []
for v in COMPARED:
    for (doc, model), m in maps[v].items():
        claims_ = [s for s in m.statements if s.claim_type not in ("OBSERVATION", "SUPPOSITION")]
        words = {s.id: content_words(s.statement) for s in claims_}
        dup = 0
        for a, b in combinations(claims_, 2):
            wa, wb = words[a.id], words[b.id]
            score = len(wa & wb) / len(wa | wb) if wa | wb else 0
            if score >= 0.4:
                dup += 1
                pairs.append({"taxonomy": v, "label": LABEL[doc], "model": model, "jaccard": round(score, 2),
                              "a": a.statement, "b": b.statement})
        rows.append({"taxonomy": v, "label": LABEL[doc], "model": model, "claims": len(claims_), "near_dup_pairs": dup,
                     "pairs_per_10_claims": round(10 * dup / max(len(claims_), 1), 2)})
dups = pd.DataFrame(rows)
pairs = pd.DataFrame(pairs)
display(dups.pivot_table(index=["label", "model"], columns="taxonomy", values=["claims", "near_dup_pairs"], aggfunc="first")
        .reindex(columns=COMPARED, level=1))
display(dups.groupby(["taxonomy", "model"])[["claims", "near_dup_pairs"]].sum().reindex(COMPARED, level=0))
pairs[pairs["taxonomy"] == "v0.11"].sort_values("jaccard", ascending=False).head(15)

claims       near_dup_pairs      
taxonomy                     v0.10 v0.11          v0.10 v0.11
label                 model                                  
BC FX Views           haiku   54.0   NaN            0.0   NaN
                      opus    89.0  70.0            1.0   0.0
Beam yields up Scotty haiku   79.0  70.0           15.0   4.0
                      opus    78.0  73.0            2.0   6.0
BofA Liquid Insight   haiku   23.0  24.0            1.0   2.0
                      opus    29.0  24.0            1.0   0.0
BofA Sept FOMC        haiku    NaN  70.0            NaN   8.0
                      opus    84.0  78.0           11.0   6.0

claims  near_dup_pairs
taxonomy model                        
v0.10    haiku     156              16
         opus      280              15
v0.11    haiku     164              14
         opus      245              12

,taxonomy,label,model,jaccard,a,b
53,v0.11,BofA Sept FOMC,opus,0.67,"If Warsh delivers the expected Jackson Hole-style hawkish press conference, the USD is likely to be supported.","If Warsh delivers the expected Jackson Hole-style hawkish press conference, risk assets are likely to be challenged."
48,v0.11,BofA Sept FOMC,haiku,0.67,The 2Y rate rises 5-10bps and the 30Y rate falls 5-10bps.,The 2Y rate falls 5bps and the 30Y rate rises 5bps.
39,v0.11,Beam yields up Scotty,opus,0.64,"Both Machine models, Boost and Forest, keep a long USDCHF pick, with the CHF as the funder.","Both Machine models, Boost and Forest, switch into a long CADCHF pick this week, with the CHF as the funder."
55,v0.11,BofA Sept FOMC,opus,0.62,2Y UST yields would rise 5-10bp and 30Y yields fall 5-10bp.,2Y UST yields would fall about 5bp and 30Y yields rise about 5bp.
56,v0.11,BofA Sept FOMC,opus,0.62,The Trump administration's choice is between tolerating modestly higher front-end rates and materially higher long-end rates.,Trump will not like the choice between higher front-end and higher long-end rates.
33,v0.11,Beam yields up Scotty,haiku,0.52,"We maintain a dollar short versus the euro, held as long EURUSD (entry 1.1595, take profit 1.1845, stop loss 1.1460).","We maintain a GBP short versus the euro, held as long EURGBP (entry 0.8540, take profit 0.8700, stop loss 0.8460)."
36,v0.11,Beam yields up Scotty,opus,0.48,"Keep the long CAD vs AUD position: short AUDCAD, entered at 0.9835 on 12 Aug 2026, take profit 0.9532, stop loss 0.9988.","Keep the long JPY vs CHF position: short CHFJPY, entered at 196.28 on 12 Aug 2026, take profit 186.19, stop loss 201.30."
51,v0.11,BofA Sept FOMC,opus,0.47,A hawkish hike should push front-end rates higher and produce a modest twist flattening of the UST curve.,"A hawkish hike, with higher front-end rates and a twist-flattening curve, will challenge BofA's earlier bias for lower belly rates."
54,v0.11,BofA Sept FOMC,opus,0.47,"The market would raise the peak of priced hikes and bring timing forward, to about 100bp in total and 60-65bp by end-2026.","The market would lower the peak of priced hikes and push out timing, to about 85bp in total and 40-45bp by end-2026."
40,v0.11,Beam yields up Scotty,opus,0.46,Value and rate differentials (carry) are the main drivers of the Machines' long USDCHF picks.,"Value, rate differentials and growth differentials drive the Machines' long CADCHF picks."


In [5]:
# The BofA case: statements about France risk being priced, or the forward-vol premium.
bofa = claims[claims["label"].str.startswith("BofA Liquid") & claims["statement"].str.contains(
    r"priced|pricing|premium|reflected", case=False, regex=True)]
bofa.sort_values(["taxonomy", "model", "statement_id"])[SHOW + ["n_evidence"]]

,taxonomy,label,model,statement_id,claim_type,sub,modality,statement,n_evidence
484,v0.10,BofA Liquid Insight,haiku,S001,VIEW,valuation,HEDGED,French fiscal and political developments look reasonably priced for now in FX.,1
505,v0.10,BofA Liquid Insight,haiku,S022,VIEW,valuation,HEDGED,The FX market has already been pricing in France-related risks to some extent.,1
508,v0.10,BofA Liquid Insight,haiku,S025,OBSERVATION,,ASSERTED,The forward volatility premium pattern is particularly evident in EUR-GBP and EUR-CHF.,1
509,v0.10,BofA Liquid Insight,haiku,S026,OBSERVATION,datapoint,ASSERTED,The steepness of the 6m3m forward volatility premium curve is 2 standard deviations above average for many EUR crosses.,1
510,v0.10,BofA Liquid Insight,haiku,S027,VIEW,valuation,HEDGED,Investors are already paying a historically large premium for volatility around the French election.,1
517,v0.10,BofA Liquid Insight,haiku,S034,VIEW,valuation,HEDGED,Other markets suggest France-related risks are at least partly priced.,1
530,v0.10,BofA Liquid Insight,opus,S001,VIEW,valuation,HEDGED,"French fiscal and political risks look reasonably, at least partly, priced in FX for now.",4
549,v0.10,BofA Liquid Insight,opus,S020,VIEW,valuation,HEDGED,"Risks around the French presidential election seem already reflected in FX options pricing, with a forward vol premium for the election.",2
552,v0.10,BofA Liquid Insight,opus,S023,OBSERVATION,,ASSERTED,"The 6m3m forward vol premium peak is particularly evident in EUR crosses, including EUR-GBP and EUR-CHF.",1
554,v0.10,BofA Liquid Insight,opus,S025,OBSERVATION,datapoint,ASSERTED,Z-scores of the forward volatility premium since 2015 show the steepness of the curve in 6m3m space is 2 standard deviations above average for many EUR cros...,1


### Observations

- **Near-duplicates fall modestly.** Opus went from 0.54 to 0.49 pairs per 10 claims (15 → 12). Haiku on the comparable docs went from 1.57 to 0.64 (16 → 6 pairs), driven by Beam (15 → 4). However, Beam opus (2 → 6) and FOMC haiku (8 pairs, new) are not clean.
- **Most v0.11 top pairs are false positives:**
  - Opposite scenario branches, e.g. "2Y rises / 30Y falls" vs "2Y falls / 30Y rises", or "raise the hike peak" vs "lower the hike peak".
  - Parallel trades, e.g. EURUSD vs EURGBP, or the Machine USDCHF vs CADCHF picks.
- **Genuine duplicates remain:**
  - FOMC opus: "the administration's choice is between front-end and long-end rates" vs "Trump will not like that choice".
  - FOMC opus: "a hawkish hike pushes the front end higher" vs "a hawkish hike challenges the belly bias".
  - FOMC haiku: "Fed will hike 25bp" vs "Warsh likely to deliver a hawkish hike".
- **BofA "France risk is priced":**
  - Haiku: 4 VIEWs plus 2 OBSERVATIONs became 4 VIEWs. S001 merges two v0.10 views, but the proxy (S012) and "other markets" (S019) stay separate.
  - Opus: 4 VIEWs plus 2 OBSERVATIONs became 5 VIEWs. S001 merges FX pricing with the proxy, "other markets" is split into rates (S020) and equities (S023), and options pricing is still two claims (S015, S016).
  - Overall, the BofA case is only partly fixed.
- **Opus puts data inside the claim text.** S016 includes the "2 std dev" steepness and S020 the OAT-Bund level. The observation reappears as a second sentence of the claim rather than only as evidence.

## 3. Claim types and hierarchy

In [6]:
display(
    pd.crosstab([claims["taxonomy"], claims["model"]], claims["claim_type"], margins=True)
    .reindex(columns=[*TYPES, "All"], fill_value=0)
)


def hierarchy_stats(v):
    rows = []
    for (doc, model), m in maps[v].items():
        types = {s.id: s.claim_type for s in m.statements}
        edges, sedges = argument_edges(m), scenario_edges(m)
        core_of = {a.core_claim for a in m.arguments}
        linked = {x for e in edges for x in (e[0], e[2])} | {e[2] for e in sedges}
        members = lambda a: [sid for r in ("premises", "supporting", "conditions", "qualifiers", "concessions", "risks",
                                           "catalysts", "implications", "alternatives") for sid in getattr(a, r)]
        recs = [a for a in m.arguments if types[a.core_claim] == "RECOMMENDATION"]
        rows.append({
            "taxonomy": v, "label": LABEL[doc], "model": model, "arguments": len(m.arguments),
            "claims_per_arg": round(sum(types[s] != "OBSERVATION" for s in types) / len(m.arguments), 1),
            "arg_links": len(edges), "scenario_links": len(sedges),
            "isolated_args": sum(a.id not in linked for a in m.arguments),
            "bare_args": sum(not members(a) for a in m.arguments),
            "recs": len(recs),
            "recs_motivated": sum(any(s in core_of for s in a.premises + a.supporting) for a in recs),
        })
    return pd.DataFrame(rows)


hierarchy = pd.concat([hierarchy_stats(v) for v in COMPARED], ignore_index=True)
display(hierarchy.groupby(["taxonomy", "model"]).sum(numeric_only=True).drop(columns="claims_per_arg")
        .reindex(COMPARED, level=0))
hierarchy.pivot_table(index=["label", "model"], columns="taxonomy",
                      values=["arguments", "arg_links", "isolated_args", "bare_args", "recs", "recs_motivated"],
                      aggfunc="first").reindex(columns=COMPARED, level=1)

claim_type      RECOMMENDATION  PREDICTION  CAUSAL  VIEW  OBSERVATION  \
taxonomy model                                                          
v0.10    haiku              18          40      40    58           91   
         opus               14         104      59   103          122   
v0.11    haiku              16          58      29    61            0   
         opus               13          89      54    89            0   
All                         61         291     182   311          213   

claim_type      SUPPOSITION   All  
taxonomy model                     
v0.10    haiku           11   258  
         opus            29   431  
v0.11    haiku           15   179  
         opus            26   271  
All                      81  1139

arguments  arg_links  scenario_links  isolated_args  \
taxonomy model                                                        
v0.10    haiku         55         25               0             22   
         opus          71         42              13             18   
v0.11    haiku         52         25               6             19   
         opus          75         39              10             28   

                bare_args  recs  recs_motivated  
taxonomy model                                   
v0.10    haiku          6    18               8  
         opus           7    13               6  
v0.11    haiku          2    15               3  
         opus           4    13               4

arg_links       arguments       bare_args        \
taxonomy                        v0.10 v0.11     v0.10 v0.11     v0.10 v0.11   
label                 model                                                   
BC FX Views           haiku       4.0   NaN      14.0   NaN       1.0   NaN   
                      opus       11.0  10.0      15.0  17.0       0.0   0.0   
Beam yields up Scotty haiku      15.0   8.0      32.0  26.0       4.0   2.0   
                      opus       14.0  17.0      26.0  29.0       4.0   3.0   
BofA Liquid Insight   haiku       6.0   7.0       9.0  10.0       1.0   0.0   
                      opus        5.0   2.0       9.0   6.0       2.0   0.0   
BofA Sept FOMC        haiku       NaN  10.0       NaN  16.0       NaN   0.0   
                      opus       12.0  10.0      21.0  23.0       1.0   1.0   

                            isolated_args        recs       recs_motivated  \
taxonomy                            v0.10 v0.11 v0.10 v0.11          v0.10   
label                 model                                                  
BC FX Views           haiku           7.0   NaN   0.0   NaN            0.0   
                      opus            2.0   5.0   0.0   0.0            0.0   
Beam yields up Scotty haiku          13.0  15.0  16.0  13.0            7.0   
                      opus            8.0   9.0  11.0  11.0            5.0   
BofA Liquid Insight   haiku           2.0   1.0   2.0   2.0            1.0   
                      opus            1.0   3.0   2.0   2.0            1.0   
BofA Sept FOMC        haiku           NaN   3.0   NaN   0.0            NaN   
                      opus            7.0  11.0   0.0   0.0            0.0   

                                   
taxonomy                    v0.11  
label                 model        
BC FX Views           haiku   NaN  
                      opus    0.0  
Beam yields up Scotty haiku   2.0  
                      opus    3.0  
BofA Liquid Insight   haiku   1.0  
                      opus    1.0  
BofA Sept FOMC        haiku   0.0  
                      opus    0.0

### Observations

- **Opus claim mix** (v0.10 → v0.11):

  | Type | v0.10 | v0.11 |
  |---|---|---|
  | PREDICTION | 104 | 89 |
  | VIEW | 103 | 89 |
  | CAUSAL | 59 | 54 |
  | SUPPOSITION | 29 | 26 |
  | RECOMMENDATION | 14 | 13 |

  Observations are 0, as intended. Non-observation claims fell 12% (309 → 271), consistent with merging.
- **Fewer bare arguments:** haiku 6 → 2, opus 7 → 4. This was a warning, not an error.
- **More isolated arguments for opus:** 18 → 28 (BC FX 2 → 5, BofA LI 1 → 3, FOMC 9 → 11). Opus argument links fell slightly (42 → 39).
- **`recs_motivated` fell** (haiku 8 → 3, opus 6 → 4), but this does not mean recommendations lost support. Every v0.11 recommendation has at least one premise, whereas three v0.10 opus recommendations had none. The premises are now leaf claims inside the same argument rather than the core claims of other arguments. Arguments are therefore more self-contained but less cross-linked, which also explains the rise in isolated arguments.

## 4. Coverage: v0.10 claims in v0.11

Each v0.10 map is aligned to the v0.11 map of the same sample and model, and claims (not observations) are compared. A v0.10 claim with no match may have been merged into another v0.11 claim, which is intended, rather than lost. The unmatched must-keep claims are listed for review.

In [ ]:
from incolor.argumentation.evaluate import span_overlap

# matched: a one-to-one statement match. covered: some v0.11 statement cites the v0.10 claim's evidence,
# so a claim merged into another still counts.
alignment_path = RUN_DIRS["v0.11"] / "claims_analysis" / "v10_alignment.csv"
current = sorted(f"{d}/{m}" for d, m in maps["v0.11"])
if alignment_path.exists() and sorted(pd.read_csv(alignment_path, encoding="utf-8-sig")["run"].unique()) == current:
    align = pd.read_csv(alignment_path, encoding="utf-8-sig")
else:
    rows = []
    for (doc, model), m11 in maps["v0.11"].items():
        m10 = maps["v0.10"].get((doc, model))
        if m10 is None:
            continue
        new = {s.id: s for s in m11.statements}
        mapping = match_statements(m10, m11)
        ev10 = {e.id: e.text for e in m10.evidence}
        cited11 = [e.text for e in m11.evidence if any(e.id in s.evidence for s in m11.statements)]
        for s in m10.statements:
            n = new.get(mapping.get(s.id))
            covered = all(any(span_overlap(ev10[eid], t) >= 0.8 for t in cited11) for eid in s.evidence)
            rows.append({
                "run": f"{doc}/{model}", "doc": doc, "model": model, "v10_id": s.id, "v10_type": s.claim_type,
                "v10_sub": "+".join(s.subkinds), "v10_statement": s.statement,
                "v11_id": n.id if n else None, "v11_type": n.claim_type if n else None,
                "v11_statement": n.statement if n else None, "covered": covered,
            })
    align = pd.DataFrame(rows)
    align.to_csv(alignment_path, index=False, encoding="utf-8-sig")
align["label"] = align["doc"].map(LABEL)
align["matched"] = align["v11_id"].notna()
v10_claims = align[align["v10_type"] != "OBSERVATION"].copy()
v10_claims["must_keep"] = v10_claims["v10_type"].isin(["RECOMMENDATION", "PREDICTION"]) | v10_claims["v10_sub"].isin(
    ["valuation", "positioning"])
display(v10_claims.groupby(["label", "model"])[["matched", "covered"]].mean().round(2)
        .join(v10_claims.groupby(["label", "model"]).size().rename("v10_claims")))
display(v10_claims.groupby("v10_type")[["matched", "covered"]].mean().round(2)
        .join(v10_claims.groupby("v10_type").size().rename("n")))
obs = align[align["v10_type"] == "OBSERVATION"]
print(f"v0.10 observations whose evidence is still cited in v0.11: {obs['covered'].mean():.0%} of {len(obs)}")
print("v0.10 must-keep claims whose evidence is not cited in v0.11 (candidate losses)")
v10_claims[v10_claims["must_keep"] & ~v10_claims["covered"]][["label", "model", "v10_type", "v10_sub", "v10_statement"]]

matched  covered  v10_claims
label                 model                              
BC FX Views           opus      0.72     0.98          99
Beam yields up Scotty haiku     0.68     0.98          81
                      opus      0.66     0.98          80
BofA Liquid Insight   haiku     0.84     1.00          25
                      opus      0.67     0.83          30
BofA Sept FOMC        opus      0.82     0.95         100

,matched,covered,n
v10_type,,,
CAUSAL,0.80,0.95,80
PREDICTION,0.75,0.96,120
RECOMMENDATION,0.88,0.88,32
SUPPOSITION,0.82,1.00,33
VIEW,0.62,0.98,150


v0.10 observations whose evidence is still cited in v0.11: 81% of 188
v0.10 must-keep claims whose evidence is not cited in v0.11 (candidate losses)


,label,model,v10_type,v10_sub,v10_statement
148,Beam yields up Scotty,haiku,RECOMMENDATION,trade_idea,"Maintain a short GBP versus EUR via long EURGBP, open since 12 Aug 2026 at 0.8540 with take profit 0.8700 and stop loss 0.8460."
150,Beam yields up Scotty,haiku,RECOMMENDATION,trade_idea,"Maintain a short USD versus EUR via long EURUSD, open since 19 Aug 2026 at 1.1595 with take profit 1.1845 and stop loss 1.1460."
272,Beam yields up Scotty,opus,RECOMMENDATION,trade_idea,"Remain underweight the USD in the portfolio, held as a long EURUSD position (short USD vs EUR) entered 19 Aug 2026 at 1.1595, with take profit 1.1845 and st..."
277,Beam yields up Scotty,opus,RECOMMENDATION,trade_idea,"The long CADCHF position (entered 12 Aug 2026 at 0.5829) was stopped out and closed at 0.5736 on 24 Aug 2026, a return of -1.63%."
457,BofA Liquid Insight,opus,PREDICTION,event,"Budget/fiscal issues dominate France in the near term, and France may prove more of a 2027 story than a late-2026 story for the EUR, with the elections like..."
460,BofA Liquid Insight,opus,PREDICTION,event,The next French President could dissolve parliament and call snap parliamentary elections.
463,BofA Liquid Insight,opus,VIEW,valuation,"Risks around the French presidential election seem already reflected in FX options pricing, with a forward vol premium for the election."
523,BofA Sept FOMC,opus,PREDICTION,event,"The Fed will include the PCE methodology changes due on September 30 in its projections, though BofA's conviction on this is low."
529,BofA Sept FOMC,opus,PREDICTION,directional,"The Fed's unemployment rate medians will move down as a mark-to-market, despite the higher policy path."
536,BofA Sept FOMC,opus,PREDICTION,event,"Asked why the Fed did not hike in July, Warsh will say that progress on underlying inflation has been insufficient."


### Observations

- **One-to-one matching is 0.66–0.84 per run, but evidence coverage is 0.83–1.00.** Most unmatched v0.10 claims were merged, not dropped.
  - VIEW has the lowest match rate (0.62) but 0.98 coverage, so it is where most merging happened.
  - RECOMMENDATION has 0.88 on both measures.
- **About 81% of the v0.10 observation evidence is still cited by v0.11 claims.** The other 19% is data that supported no claim.
- **Only 2 of the 10 candidate must-keep losses are real.** I searched the v0.11 maps for each one. Eight are present but cite differently worded evidence, so the 0.8 span-overlap test misses them:
  - the EURGBP and EURUSD trades in Beam (haiku and opus)
  - BofA options pricing (now S015) and "2027 story" (now S004)
  - FOMC PCE methodology (S032), unemployment medians (S038) and "Warsh: insufficient progress" (S046)
- **The two real losses are both defensible:**
  - Beam opus: "CADCHF stopped out at −1.63%" is a closed-trade result, not a live recommendation.
  - BofA opus: "the next President could dissolve parliament" is now excluded as `NON_CLAIM` (institutional background).
- **No genuine must-keep loss.** For recommendations that cite a table row in one version and prose in the other, the probe needs a looser match or a match on statement text.

## 5. Model invariance: haiku vs opus

Statements are matched between the haiku and opus maps of each sample. In v0.10, observations are left out on both sides so the comparison covers claims only.

In [8]:
def claims_only(m):
    return m.model_copy(update={"statements": [s for s in m.statements if s.claim_type != "OBSERVATION"]})


rows = []
for v in COMPARED:
    for doc in DOCS:
        if not all((doc, model) in maps[v] for model in MODELS):
            continue
        a, b = claims_only(maps[v][(doc, "haiku")]), claims_only(maps[v][(doc, "opus")])
        mapping = match_statements(a, b)
        sa, sb = {s.id: s for s in a.statements}, {s.id: s for s in b.statements}
        pairs_ = [(sa[x], sb[y]) for x, y in mapping.items()]
        n = len(pairs_)
        share = lambda test, pairs_=pairs_, n=n: round(sum(test(x, y) for x, y in pairs_) / n, 2) if n else None
        rows.append({
            "taxonomy": v, "label": LABEL[doc], "haiku_claims": len(sa), "opus_claims": len(sb), "matched": n,
            "claim_overlap_f1": round(2 * n / (len(sa) + len(sb)), 2),
            "type": share(lambda x, y: x.claim_type == y.claim_type),
            "subkinds": share(lambda x, y: set(x.subkinds) == set(y.subkinds)),
            "modality": share(lambda x, y: x.modality == y.modality),
        })
agreement = pd.DataFrame(rows)
agreement.pivot_table(index="label", columns="taxonomy",
                      values=["haiku_claims", "opus_claims", "claim_overlap_f1", "type", "subkinds"],
                      aggfunc="first").reindex(columns=COMPARED, level=1)

claim_overlap_f1       haiku_claims       opus_claims  \
taxonomy                         v0.10 v0.11        v0.10 v0.11       v0.10   
label                                                                         
BC FX Views                       0.69   NaN         61.0   NaN        99.0   
Beam yields up Scotty             0.82  0.68         81.0  72.0        80.0   
BofA Liquid Insight               0.76  0.81         25.0  26.0        30.0   
BofA Sept FOMC                     NaN  0.56          NaN  81.0         NaN   

                            subkinds        type        
taxonomy              v0.11    v0.10 v0.11 v0.10 v0.11  
label                                                   
BC FX Views             NaN     0.65   NaN  0.87   NaN  
Beam yields up Scotty  75.0     0.86  0.84  0.94  0.92  
BofA Liquid Insight    26.0     0.86  0.90  0.90  1.00  
BofA Sept FOMC         91.0      NaN  0.75   NaN  0.85

### Observations

- **Only Beam and BofA LI can be compared across versions,** because haiku failed BC FX in v0.11 and FOMC in v0.10.

  | Sample | Claim-overlap F1 (v0.10 → v0.11) | Type agreement on matched pairs |
  |---|---|---|
  | Beam | 0.82 → 0.68 | 0.94 → 0.92 |
  | BofA LI | 0.76 → 0.81 | 0.90 → 1.00 |

- **No clear improvement in agreement.** The two models now produce similar numbers of claims (Beam 72 vs 75), but merge differently. For example, haiku spreads the USD short over S001, S002 and S037, while opus folds it into one claim (S002). One-to-one matching penalises this.
- **FOMC has the lowest overlap in v0.11 (F1 0.56),** although type agreement is still 0.85.
- **When the models extract the same claim, they type it consistently:** type agreement is 0.85–1.00 and subkind agreement 0.75–0.90. The disagreement is about which claims to extract and how to group them.

## 6. Grounding: claims and their evidence

With observations dropped, claims should cite the data as evidence. Measured per claim: the number of evidence quotes, and whether any quote contains a number.

In [9]:
non_obs = claims[claims["claim_type"] != "OBSERVATION"].copy()
non_obs["numeric_evidence"] = non_obs["evidence_texts"].map(lambda texts: any(re.search(r"\d", t) for t in texts))
non_obs["evidence_chars"] = non_obs["evidence_texts"].map(lambda texts: sum(len(t) for t in texts))
display(non_obs.groupby(["taxonomy", "model"]).agg(
    claims=("statement_id", "size"), mean_evidence=("n_evidence", "mean"), numeric_evidence=("numeric_evidence", "mean"),
    median_evidence_chars=("evidence_chars", "median"), mean_statement_chars=("statement", lambda s: s.str.len().mean()),
).round(2).reindex(COMPARED, level=0))
# Claims citing the most evidence in v0.11: merged points, or over-long grounding?
non_obs[non_obs["taxonomy"] == "v0.11"].nlargest(8, "n_evidence")[SHOW + ["n_evidence"]]

claims  mean_evidence  numeric_evidence  \
taxonomy model                                            
v0.10    haiku     167           1.15              0.37   
         opus      309           1.27              0.43   
v0.11    haiku     179           1.33              0.42   
         opus      271           1.71              0.50   

                median_evidence_chars  mean_statement_chars  
taxonomy model                                               
v0.10    haiku                  156.0                 80.83  
         opus                   166.0                102.49  
v0.11    haiku                  156.0                 80.74  
         opus                   186.0                123.83

,taxonomy,label,model,statement_id,claim_type,sub,modality,statement,n_evidence
704,v0.11,BC FX Views,opus,S016,VIEW,positioning,ASSERTED,"Household JPY selling through NISA investment trusts is accelerating: overseas investment is running faster than in 2024 and 2025, with no slowdown in H2 an...",10
783,v0.11,Beam yields up Scotty,haiku,S016,VIEW,positioning,ASSERTED,The US is no longer the only game in town for long-term debt sales.,6
841,v0.11,Beam yields up Scotty,opus,S002,RECOMMENDATION,trade_idea,ASSERTED,"Stay underweight the USD and keep the short USD position against the EUR, including into Jackson Hole: long EURUSD, entered at 1.1595 on 19 Aug 2026, take p...",6
941,v0.11,BofA Liquid Insight,opus,S001,VIEW,valuation,HEDGED,"French fiscal and political risks look reasonably priced for now in FX, and to some extent the FX market has already been pricing them in. BofA's simple pro...",6
1052,v0.11,BofA Sept FOMC,opus,S005,PREDICTION,event,HEDGED,"Warsh is likely to deliver a hawkish hike. Sept FOMC communications will lean hawkish, and his press conference will resemble Jackson Hole Warsh rather than...",6
1064,v0.11,BofA Sept FOMC,opus,S017,VIEW,event,ASSERTED,"In FOMC OIS history since 2007, the Fed has never held with hike pricing this high (22bp priced against a previous maximum of about 10bp), so a hold now wou...",6
703,v0.11,BC FX Views,opus,S015,PREDICTION,directional,HEDGED,Continued large-scale M&A and overseas investment through the new NISA scheme mean that structural JPY-selling pressure will persist. Excluding the temporar...,5
761,v0.11,BC FX Views,opus,S073,VIEW,event,HEDGED,"The US share of the end-July intervention was likely small and probably funded from the ESF's foreign currency holdings (about USD4.4bn in JPY and EUR), so ...",5


### Observations

- **Claims now cite more evidence, and more of it contains numbers:**

  | Measure | Haiku (v0.10 → v0.11) | Opus (v0.10 → v0.11) |
  |---|---|---|
  | Evidence quotes per claim | 1.15 → 1.33 | 1.27 → 1.71 |
  | Claims with numeric evidence | 37% → 42% | 43% → 50% |

  This is the intended effect: the data is now attached to claims as evidence.
- **Opus statements are longer: 102 → 124 characters on average (+21%), while haiku is flat (81).** The most-cited opus claims are multi-sentence:
  - BC FX S016 cites 10 quotes.
  - FOMC S005 and S017, and BofA S001, cite 6 each.
  - In each case the claim merges a point with the data behind it.
- **Grounding improved, but opus over-merges.** One claim can carry several assertions plus the data, which goes against the atomicity rule ("distinct if one could be true while the other is false"). The prompt should say that data belongs in the evidence and not in the statement text.

## 7. Suppositions, scenarios and recommendations

In [10]:
CONDITIONAL = r"\b(?:if|unless|should the|in the event|assuming|after a|in a \w+ scenario)\b"
sup = claims[claims["claim_type"] == "SUPPOSITION"]
display(pd.crosstab([sup["taxonomy"], sup["model"]], sup["role_set"].fillna("") + " " + sup["scenario_slots"].fillna("")))
cond = claims[(claims["claim_type"] != "SUPPOSITION") & claims["statement"].str.contains(CONDITIONAL, case=False, regex=True)]
print("Claims that still carry a conditional clause")
display(pd.crosstab(cond["taxonomy"], cond["model"]))
rows = []
for v in COMPARED:
    for (doc, model), m in maps[v].items():
        refs = Counter(role for _, role, _ in scenario_edges(m))
        rows.append({"taxonomy": v, "label": LABEL[doc], "model": model, "scenarios": len(m.scenarios),
                     "labels": "; ".join(s.label for s in m.scenarios), "referenced_from": dict(refs)})
display(pd.DataFrame(rows).query("scenarios > 0"))
recs = claims[claims["claim_type"] == "RECOMMENDATION"]
display(pd.crosstab([recs["taxonomy"], recs["model"]], recs["sub"]))
recs[recs["taxonomy"] == "v0.11"][SHOW + ["attribution", "role_set"]]

col_0            SC001:assumptions   SC002:assumptions   SC003:assumptions  \
taxonomy model                                                               
v0.10    haiku                   0                   0                   0   
         opus                    2                   2                   1   
v0.11    haiku                   2                   2                   2   
         opus                    1                   0                   0   

col_0            SC003:assumptions|SC005:assumptions   SC004:assumptions  \
taxonomy model                                                             
v0.10    haiku                                     0                   0   
         opus                                      0                   3   
v0.11    haiku                                     0                   1   
         opus                                      1                   2   

col_0            SC005:assumptions   SC006:assumptions  conditions   \
taxonomy model                                                        
v0.10    haiku                   0                   0           11   
         opus                    1                   3           17   
v0.11    haiku                   2                   0            6   
         opus                    2                   0           17   

col_0           conditions SC002:assumptions|SC005:assumptions  \
taxonomy model                                                   
v0.10    haiku                                               0   
         opus                                                0   
v0.11    haiku                                               0   
         opus                                                1   

col_0           conditions SC004:assumptions  
taxonomy model                                
v0.10    haiku                             0  
         opus                              0  
v0.11    haiku                             0  
         opus                              2

Claims that still carry a conditional clause


model,haiku,opus
taxonomy,,
v0.10,5,5
v0.11,4,9


,taxonomy,label,model,scenarios,labels,referenced_from
1,v0.10,BC FX Views,opus,2,GPIF 5-10pp reallocation to domestic assets; Reallocation via FX-hedged foreign bonds,"{'alternatives': 1, 'implications': 1, 'risks': 1}"
6,v0.10,BofA Sept FOMC,opus,6,Hawkish presser; Dovish presser; Fed hold; Bullish USD case; Bearish USD case: Fed hold; Bearish USD case: dovish or muddled hike,"{'risks': 2, 'implications': 5, 'alternatives': 3}"
12,v0.11,BofA Sept FOMC,haiku,5,Hawkish presser; Dovish presser; Fed hold; Bullish USD case; Bearish USD case,"{'alternatives': 3, 'conditions': 1, 'risks': 2}"
13,v0.11,BofA Sept FOMC,opus,5,Hawkish presser; Dovish presser; Fed hold; Bullish USD case; Bearish USD case,"{'alternatives': 3, 'implications': 4, 'risks': 3}"


sub             rating  trade_idea
taxonomy model                    
v0.10    haiku       1          17
         opus        0          14
v0.11    haiku       0          16
         opus        0          13

,taxonomy,label,model,statement_id,claim_type,sub,modality,statement,attribution,role_set
768,v0.11,Beam yields up Scotty,haiku,S001,RECOMMENDATION,trade_idea,ASSERTED,We remain underweight the USD in our portfolio.,NaN,core_claim
769,v0.11,Beam yields up Scotty,haiku,S002,RECOMMENDATION,trade_idea,ASSERTED,Sit short of dollars into the Jackson Hole gathering.,NaN,qualifiers
795,v0.11,Beam yields up Scotty,haiku,S028,RECOMMENDATION,trade_idea,ASSERTED,"We maintain a CAD long versus the AUD, held as short AUDCAD (entry 0.9835, take profit 0.9532, stop loss 0.9988).",NaN,core_claim
800,v0.11,Beam yields up Scotty,haiku,S033,RECOMMENDATION,trade_idea,ASSERTED,"We keep a JPY long versus the CHF, held as short CHFJPY (entry 196.28, take profit 186.19, stop loss 201.30).",NaN,core_claim
804,v0.11,Beam yields up Scotty,haiku,S037,RECOMMENDATION,trade_idea,ASSERTED,"We maintain a dollar short versus the euro, held as long EURUSD (entry 1.1595, take profit 1.1845, stop loss 1.1460).",NaN,core_claim|implications
806,v0.11,Beam yields up Scotty,haiku,S039,RECOMMENDATION,trade_idea,ASSERTED,"We maintain a GBP short versus the euro, held as long EURGBP (entry 0.8540, take profit 0.8700, stop loss 0.8460).",NaN,core_claim
810,v0.11,Beam yields up Scotty,haiku,S043,RECOMMENDATION,trade_idea,ASSERTED,We maintain a neutral NZD stance.,NaN,core_claim
814,v0.11,Beam yields up Scotty,haiku,S047,RECOMMENDATION,trade_idea,ASSERTED,We stay neutral on SEK this week.,NaN,core_claim
818,v0.11,Beam yields up Scotty,haiku,S051,RECOMMENDATION,trade_idea,ASSERTED,We adopt a neutral NOK stance for now.,NaN,core_claim
826,v0.11,Beam yields up Scotty,haiku,S059,RECOMMENDATION,trade_idea,ASSERTED,"The Machine is short CHF versus JPY, through short CHFJPY.",Machine,core_claim


### Observations

- **Haiku now builds FOMC scenarios.** It produced 5 scenarios with labels identical to opus's (Hawkish presser, Dovish presser, Fed hold, Bullish USD case, Bearish USD case). Some of its suppositions are scenario assumptions, which none were in v0.10.
- **Opus scenarios:**
  - FOMC: the two bearish-USD cases from v0.10 are merged, giving 6 → 5 scenarios.
  - BC FX: the v0.10 GPIF scenarios (2) are gone.
- **Supposition counts:** haiku 11 → 15, opus 29 → 26.
- **Conditional clauses in non-suppositions:** haiku 5 → 4, opus 5 → 9. Most v0.11 hits are regex false positives ("if not reversed", "even if", "after a potential snap election"). The genuine cases are:
  - Opus: "If Warsh delivers a hawkish presser, USD supported / risk challenged". These duplicate the Hawkish presser scenario and should be implications of it.
  - Haiku: "If the hawkish hike is delivered…", "Waller might dissent if…" and "If Warsh is wrong…".
- **Recommendations are stable:** haiku 18 → 16, opus 14 → 13, almost all `trade_idea`. Machine-model picks carry their attribution correctly.
- **Recommendation issues:**
  - "European equity strategists are neutral on France" is a `rating`, not a `trade_idea`. Both models make this mistake.
  - Haiku Beam still spreads the USD short over a core claim, a qualifier ("sit short of dollars into Jackson Hole") and the EURUSD trade.

## 8. Summary

| Question | Verdict |
|---|---|
| 1. Validity, cost, size | Validity is unchanged (haiku 3/4, opus 4/4). Opus is 41% cheaper, needs 1 attempt and produces 37% fewer statements. BC FX haiku still fails on verbatim evidence. |
| 2. Duplication | Somewhat lower (opus 0.54 → 0.49 pairs per 10 claims; haiku much lower on Beam). The BofA "priced" case is only partly fixed: opus still uses 5 claims. |
| 3. Hierarchy | Fewer bare arguments, and every recommendation has premises. Arguments are more self-contained, so there are more isolated arguments and fewer links to other arguments. |
| 4. Coverage | No genuine must-keep loss. 8 of the 10 flagged claims are present, and the other 2 are defensible. VIEWs are where most merging happened. |
| 5. Model agreement | Not better. Beam F1 0.82 → 0.68, BofA LI 0.76 → 0.81, FOMC 0.56. The models agree on types but group claims differently. |
| 6. Grounding | Better: more evidence per claim and more numeric evidence. Opus also writes the data into the claim text, giving longer, multi-assertion statements. |
| 7. Suppositions, scenarios | Haiku now builds FOMC scenarios that match opus's. A few conditional predictions still duplicate scenario branches. |

**Next steps**
- Tell the prompt that data goes in the evidence only, with one assertion per statement, to curb opus over-merging.
- Fold "If X, then Y" predictions into scenario implications.
- Add a `rating` example (e.g. "neutral on France").
- Fix the coverage probe so it can match on statement text.
- Retry haiku BC FX.